<a href="https://colab.research.google.com/github/milandhore/genai/blob/default/MD_Zero_Cost_RAG_DeepSeekR1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install colab-xterm
%load_ext colabxterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 4.3 MB/s eta 0:00:00


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import threading

def start_ollama_server():
    subprocess.Popen("ollama serve", shell=True)

threading.Thread(target=start_ollama_server, daemon=True).start()

In [ ]:
!ollama pull nomic-embed-text

In [ ]:
!ollama pull deepseek-r1:7b

In [6]:
!pip -qq install langchain
!pip -qq install langchain-core
!pip -qq install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00


In [7]:
from langchain_community.llms import Ollama
llm = Ollama(model = "deepseek-r1:7b")
llm.invoke("what is the Meaning of life")

<ipython-input-7-a0bbe775058e>:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model = "deepseek-r1:7b")


'<think>\nOkay, so I\'m trying to figure out what the meaning of life means. I mean, it\'s a pretty big question and people probably think about it a lot. Let me start by just thinking about how I feel when I reflect on my own life. Sometimes I wonder why I exist or if there\'s a purpose beyond the random stuff that happens to us.\n\nI guess some people find meaning in their work. Like, if you\'re an artist or a scientist or a teacher, creating something valuable and helping others might give you a sense of purpose. But then again, maybe not everyone has such fulfilling jobs. Some people lead ordinary lives but still manage to find happiness through relationships or just having fun with friends.\n\nThen there\'s religion. A lot of people turn to faith for answers. I know that in my family, my mom talks about how she believes in something bigger than herself, like a higher power or the afterlife. It makes her feel connected and gives her some comfort. But I\'m not sure if everyone who f

In [9]:
!pip install ollama langchain beautifulsoup4 chromadb gradio -q

In [ ]:
import gradio as gr
import ollama
from bs4 import BeautifulSoup as bs
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings


In [11]:
# Load the data from the web URL
url = 'https://en.wikipedia.org/wiki/Pune'
loader = WebBaseLoader(url)
docs = loader.load()

In [12]:
# Split the loaded documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [13]:
# Create Ollama embeddings and vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

<ipython-input-13-48926a9a2b23>:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [14]:
# Define the function to call the Ollama Llama3 model
def ollama_llm(question, context):
    formatted_prompt = f"Question: {question}\n\nContext: {context}"
    response = ollama.chat(model='deepseek-r1:1.5b', messages=[{'role': 'user', 'content': formatted_prompt}])
    return response['message']['content']

In [15]:
# Define the RAG setup
retriever = vectorstore.as_retriever()
def rag_chain(question):
    retrieved_docs = retriever.invoke(question)
    formatted_context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return ollama_llm(question, formatted_context)

In [16]:
# Define the Gradio interface
def get_important_facts(question):
    return rag_chain(question)

In [17]:
# Create a Gradio app interface
iface = gr.Interface(
  fn=get_important_facts,
  inputs=gr.Textbox(lines=2, placeholder="Ask Me Anything Milan About Pune..."),
  outputs="text",
  title="No Cost RAG With DeepSeek R1",
  description="Ask questions about the provided context",
)

In [18]:
# Launch the Gradio app
iface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b78830b0259c5fa32.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
